# Standardisasi RGB


## 0. Mount Google Drive & Path Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
from pathlib import Path

val_no_tumor   = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_split_test/val/no_tumor'
val_tumor      = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_split_test/val/tumor'
train_no_tumor = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_split_test/train/no_tumor'
train_tumor    = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_split_test/train/tumor'
test_no_tumor  = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_split_test/test/no_tumor'
test_tumor     = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_split_test/test/tumor'

val_tumor_masks   = val_tumor   + '/masks'
train_tumor_masks = train_tumor + '/masks'
test_tumor_masks  = test_tumor  + '/masks'

DATASET_PATHS = {
    'val/no_tumor'      : val_no_tumor,
    'val/tumor'         : val_tumor,
    'val/tumor/masks'   : val_tumor_masks,
    'train/no_tumor'    : train_no_tumor,
    'train/tumor'       : train_tumor,
    'train/tumor/masks' : train_tumor_masks,
    'test/no_tumor'     : test_no_tumor,
    'test/tumor'        : test_tumor,
    'test/tumor/masks'  : test_tumor_masks,
}

def _is_mask_subset(subset_key: str) -> bool:
    """True jika key adalah folder mask (akhiran '/masks')."""
    return subset_key.rstrip('/').endswith('masks')


## 1. Instalasi & Import Library


In [ ]:
!pip install -q Pillow tqdm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}


## 2. Konfigurasi


In [ ]:
OUTPUT_EXT  = '.png'
OUTPUT_ROOT = '/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_preprocessed_test'


## 3. Fungsi Standardisasi RGB


In [ ]:
def standardize_to_rgb(img: Image.Image, is_mask: bool = False) -> Image.Image:
    """Konversi ke RGB; mask tetap grayscale (mode L) karena label biner."""
    if is_mask:
        if img.mode != 'L':
            img = img.convert('L')
        return img
    if img.mode != 'RGB':
        img = img.convert('RGB')
    return img


## 4. Visualisasi Contoh (Before vs After)


In [ ]:
def visualize_rgb_samples(dataset_paths: dict, n_samples: int = 4) -> None:
    """Tampilkan perbandingan citra asli vs hasil standardisasi RGB."""
    candidates = []
    for label, folder in dataset_paths.items():
        if _is_mask_subset(label):
            continue  # mask tidak divisualisasikan di sini
        folder_p = Path(folder)
        if folder_p.exists():
            candidates.extend(
                f for f in folder_p.iterdir() if f.suffix.lower() in EXTENSIONS
            )

    if not candidates:
        print('Tidak ada citra ditemukan. Periksa path dataset.')
        return

    samples = candidates[:n_samples]
    fig, axes = plt.subplots(2, len(samples), figsize=(4 * len(samples), 8))

    for col, path in enumerate(samples):
        img_before = Image.open(path)
        img_after  = standardize_to_rgb(img_before.copy())

        axes[0, col].imshow(img_before, cmap='gray' if img_before.mode == 'L' else None)
        axes[0, col].set_title(f'Before\nMode: {img_before.mode}', fontsize=9)
        axes[0, col].axis('off')

        axes[1, col].imshow(img_after)
        axes[1, col].set_title(f'After\nMode: {img_after.mode}', fontsize=9)
        axes[1, col].axis('off')

    plt.tight_layout()
    plt.show()

visualize_rgb_samples(DATASET_PATHS)


## 5. Batch Processing — Seluruh Dataset


In [ ]:
def run_rgb_pipeline(dataset_paths: dict, output_root: str) -> dict:
    """Konversi seluruh dataset ke RGB (mask -> grayscale) dan simpan sebagai PNG."""
    summary = {}
    for subset_name, src_folder in dataset_paths.items():
        src_path = Path(src_folder)
        dst_path = Path(output_root) / subset_name
        dst_path.mkdir(parents=True, exist_ok=True)

        if not src_path.exists():
            print(f'[{subset_name}] Peringatan: path tidak ditemukan, dilewati.')
            summary[subset_name] = {'success': 0, 'fail': 0, 'total': 0}
            continue

        image_files = [f for f in src_path.iterdir() if f.suffix.lower() in EXTENSIONS]
        is_mask     = _is_mask_subset(subset_name)  # mask -> grayscale, lainnya -> RGB
        n_success = n_fail = 0

        for img_file in tqdm(image_files, desc=subset_name, unit='img'):
            try:
                img     = Image.open(img_file)
                img_rgb = standardize_to_rgb(img, is_mask=is_mask)
                img_rgb.save(dst_path / (img_file.stem + OUTPUT_EXT), format='PNG')
                n_success += 1
            except Exception:
                n_fail += 1  # dihitung saja, detail tidak dicetak agar log ringkas

        summary[subset_name] = {'success': n_success, 'fail': n_fail, 'total': len(image_files)}
        if n_fail > 0:
            print(f'[{subset_name}] Peringatan: {n_fail} citra gagal diproses.')

    return summary

summary_rgb = run_rgb_pipeline(DATASET_PATHS, OUTPUT_ROOT)


In [ ]:
total   = sum(s['total'] for s in summary_rgb.values())
success = sum(s['success'] for s in summary_rgb.values())
print(f"\nTotal: {success}/{total} citra berhasil dikonversi ke RGB.")
print(f'Output: {OUTPUT_ROOT}')
